In [18]:
import os
os.makedirs("/content/notebooks", exist_ok=True)
print("Notebook directory ready")

Notebook directory ready


In [19]:
import os
import pandas as pd
import numpy as np

from pathlib import Path

print("Libraries loaded successfully")

Libraries loaded successfully


In [20]:
DATA_PATH = Path("/content/data/m5/extracted")

CALENDAR_PATH = DATA_PATH / "calendar.csv"
PRICES_PATH   = DATA_PATH / "sell_prices.csv"
SALES_PATH    = DATA_PATH / "sales_train_validation.csv"

print("Data path:", DATA_PATH)
print("Calendar:", CALENDAR_PATH)
print("Prices:", PRICES_PATH)
print("Sales:", SALES_PATH)

Data path: /content/data/m5/extracted
Calendar: /content/data/m5/extracted/calendar.csv
Prices: /content/data/m5/extracted/sell_prices.csv
Sales: /content/data/m5/extracted/sales_train_validation.csv


In [21]:
calendar = pd.read_csv(CALENDAR_PATH)
prices   = pd.read_csv(PRICES_PATH)
sales    = pd.read_csv(SALES_PATH)

print("Calendar shape:", calendar.shape)
print("Prices shape:", prices.shape)
print("Sales shape:", sales.shape)

Calendar shape: (1969, 14)
Prices shape: (6841121, 4)
Sales shape: (30490, 1919)


In [22]:
calendar_clean = calendar.copy()
prices_clean   = prices.copy()
sales_clean    = sales.copy()

print("Working copies created.")

Working copies created.


In [23]:
print("===== DUPLICATE ROW CHECK =====")
print("Calendar duplicates:", calendar_clean.duplicated().sum())
print("Prices duplicates:",   prices_clean.duplicated().sum())
print("Sales duplicates:",    sales_clean.duplicated().sum())

===== DUPLICATE ROW CHECK =====
Calendar duplicates: 0
Prices duplicates: 0
Sales duplicates: 0


In [24]:
series_id_columns = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

duplicate_series = sales_clean.duplicated(
    subset=series_id_columns
).sum()

print(
    "Duplicate product-store series:",
    duplicate_series
)

Duplicate product-store series: 0


In [25]:
print(calendar_clean["date"].dtype)

calendar_clean["date"] = pd.to_datetime(calendar_clean["date"], errors="coerce")

print(calendar_clean["date"].dtype)

object
datetime64[ns]


In [26]:
invalid_dates = calendar_clean["date"].isna().sum()
print("Invalid dates after conversion:", invalid_dates)

Invalid dates after conversion: 0


In [27]:
calendar_clean = calendar_clean.sort_values("date").reset_index(drop=True)

print("Earliest date:", calendar_clean["date"].min())
print("Latest date:",   calendar_clean["date"].max())

Earliest date: 2011-01-29 00:00:00
Latest date: 2016-06-19 00:00:00


In [28]:
sales_columns = [c for c in sales_clean.columns if c.startswith("d_")]
print("Number of daily sales columns:", len(sales_columns))

sales_clean[sales_columns] = sales_clean[sales_columns].apply(pd.to_numeric, errors="coerce")

print(sales_clean[sales_columns].dtypes.value_counts())

Number of daily sales columns: 1913
int64    1913
Name: count, dtype: int64


In [29]:
negative_sales_count = (sales_clean[sales_columns] < 0).sum().sum()
print("Negative demand observations:", negative_sales_count)

Negative demand observations: 0


In [30]:
missing_sales_count = sales_clean[sales_columns].isna().sum().sum()
print("Missing daily demand observations:", missing_sales_count)

Missing daily demand observations: 0


In [31]:
identifier_columns = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
print(sales_clean[identifier_columns].isna().sum())

item_id     0
dept_id     0
cat_id      0
store_id    0
state_id    0
dtype: int64


In [32]:
print("Price dtypes:")
print(prices_clean.dtypes)

prices_clean["sell_price"] = pd.to_numeric(prices_clean["sell_price"], errors="coerce")

print("Missing sell prices:", prices_clean["sell_price"].isna().sum())
print("Negative sell prices:", (prices_clean["sell_price"] < 0).sum())

Price dtypes:
store_id       object
item_id        object
wm_yr_wk        int64
sell_price    float64
dtype: object
Missing sell prices: 0
Negative sell prices: 0


In [33]:
prices_clean = (
    prices_clean
    .sort_values(["store_id", "item_id", "wm_yr_wk"])
    .reset_index(drop=True)
)

prices_clean.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,FOODS_1_001,11101,2.0
1,CA_1,FOODS_1_001,11102,2.0
2,CA_1,FOODS_1_001,11103,2.0
3,CA_1,FOODS_1_001,11104,2.0
4,CA_1,FOODS_1_001,11105,2.0


In [34]:
print("First day IDs:", calendar_clean["d"].head().tolist())
print("Last day IDs:",  calendar_clean["d"].tail().tolist())

print("Duplicate calendar day IDs:", calendar_clean["d"].duplicated().sum())

First day IDs: ['d_1', 'd_2', 'd_3', 'd_4', 'd_5']
Last day IDs: ['d_1965', 'd_1966', 'd_1967', 'd_1968', 'd_1969']
Duplicate calendar day IDs: 0


In [35]:
PROCESSED_PATH = Path("/content/data/m5/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

calendar_clean.to_csv(PROCESSED_PATH / "calendar_clean.csv", index=False)
prices_clean.to_csv(PROCESSED_PATH / "sell_prices_clean.csv", index=False)
sales_clean.to_csv(PROCESSED_PATH / "sales_train_validation_clean.csv", index=False)

print("Cleaned datasets saved.")

Cleaned datasets saved.


In [36]:
print(list(PROCESSED_PATH.iterdir()))

[PosixPath('/content/data/m5/processed/sell_prices_clean.csv'), PosixPath('/content/data/m5/processed/calendar_clean.csv'), PosixPath('/content/data/m5/processed/sales_train_validation_clean.csv')]


In [45]:
from pathlib import Path

src_path = Path("/content/src")
src_path.mkdir(parents=True, exist_ok=True)

preprocessing_code = '''
from pathlib import Path
import pandas as pd


def load_raw_data(data_path):
    """
    Load the M5 raw calendar, price, and sales datasets.
    """
    data_path = Path(data_path)

    calendar = pd.read_csv(
        data_path / "calendar.csv"
    )

    prices = pd.read_csv(
        data_path / "sell_prices.csv"
    )

    sales = pd.read_csv(
        data_path / "sales_train_validation.csv"
    )

    return calendar, prices, sales


def clean_calendar(calendar):
    """
    Clean and validate calendar data.
    """
    df = calendar.copy()

    df = df.drop_duplicates().reset_index(drop=True)

    df["date"] = pd.to_datetime(
        df["date"],
        errors="coerce"
    )

    df = (
        df
        .sort_values("date")
        .reset_index(drop=True)
    )

    return df


def clean_prices(prices):
    """
    Clean and validate selling-price data.
    """
    df = prices.copy()

    df = df.drop_duplicates().reset_index(drop=True)

    df["sell_price"] = pd.to_numeric(
        df["sell_price"],
        errors="coerce"
    )

    df = (
        df
        .sort_values(
            ["store_id", "item_id", "wm_yr_wk"]
        )
        .reset_index(drop=True)
    )

    return df


def clean_sales(sales):
    """
    Clean and validate sales data.
    """
    df = sales.copy()

    df = df.drop_duplicates().reset_index(drop=True)

    sales_columns = [
        col for col in df.columns
        if col.startswith("d_")
    ]

    df[sales_columns] = (
        df[sales_columns]
        .apply(pd.to_numeric, errors="coerce")
    )

    return df
'''

file_path = src_path / "preprocessing.py"

file_path.write_text(
    preprocessing_code,
    encoding="utf-8"
)

print("Created:", file_path)
print("File exists:", file_path.exists())

Created: /content/src/preprocessing.py
File exists: True


In [47]:
import os
import sys

print("Current working directory:")
print(os.getcwd())

print("\nPython import paths containing /content:")
for path in sys.path:
    if "/content" in path:
        print("-", path)

print("\nFiles in /content/src:")
print(os.listdir("/content/src"))

print("\npreprocessing.py exists:")
print(os.path.isfile("/content/src/preprocessing.py"))

Current working directory:
/content

Python import paths containing /content:
- /content
- /content/src

Files in /content/src:
['preprocessing.py']

preprocessing.py exists:
True


In [48]:
import sys
import importlib

sys.path.insert(0, "/content/src")

importlib.invalidate_caches()

import preprocessing

print("preprocessing module loaded successfully")
print("Module location:", preprocessing.__file__)

preprocessing module loaded successfully
Module location: /content/src/preprocessing.py


In [50]:
from preprocessing import (
    clean_calendar,
    clean_prices,
    clean_sales
)

print("All cleaning functions imported successfully")

All cleaning functions imported successfully


In [51]:
calendar_clean = clean_calendar(calendar)
prices_clean = clean_prices(prices)
sales_clean = clean_sales(sales)

print("Cleaning functions executed successfully.")
print()
print("Original vs cleaned shapes:")
print("Calendar:", calendar.shape, "→", calendar_clean.shape)
print("Prices:  ", prices.shape, "→", prices_clean.shape)
print("Sales:   ", sales.shape, "→", sales_clean.shape)

Cleaning functions executed successfully.

Original vs cleaned shapes:
Calendar: (1969, 14) → (1969, 14)
Prices:   (6841121, 4) → (6841121, 4)
Sales:    (30490, 1919) → (30490, 1919)


In [49]:
print("===== CLEANING VALIDATION =====")

print("Calendar shape:", calendar_clean.shape)
print("Prices shape:", prices_clean.shape)
print("Sales shape:", sales_clean.shape)

print("\nCalendar duplicates:",
      calendar_clean.duplicated().sum())

print("Price duplicates:",
      prices_clean.duplicated().sum())

print("Sales duplicates:",
      sales_clean.duplicated().sum())

print("\nMissing sales values:",
      sales_clean[sales_columns].isna().sum().sum())

print("Negative sales values:",
      (sales_clean[sales_columns] < 0).sum().sum())

print("\nInvalid calendar dates:",
      calendar_clean["date"].isna().sum())

print("\nCleaning validation completed.")

===== CLEANING VALIDATION =====
Calendar shape: (1969, 14)
Prices shape: (6841121, 4)
Sales shape: (30490, 1919)

Calendar duplicates: 0
Price duplicates: 0
Sales duplicates: 0

Missing sales values: 0
Negative sales values: 0

Invalid calendar dates: 0

Cleaning validation completed.


In [52]:
from pathlib import Path

PROCESSED_PATH = Path("/content/data/m5/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

calendar_clean.to_csv(
    PROCESSED_PATH / "calendar_clean.csv",
    index=False
)

prices_clean.to_csv(
    PROCESSED_PATH / "sell_prices_clean.csv",
    index=False
)

sales_clean.to_csv(
    PROCESSED_PATH / "sales_train_validation_clean.csv",
    index=False
)

print("Cleaned datasets saved successfully.")
print()
print("Saved files:")

for file in PROCESSED_PATH.iterdir():
    print("-", file.name)

Cleaned datasets saved successfully.

Saved files:
- sell_prices_clean.csv
- calendar_clean.csv
- sales_train_validation_clean.csv


In [53]:
print("===== SAVED FILE VALIDATION =====")

calendar_check = pd.read_csv(
    PROCESSED_PATH / "calendar_clean.csv"
)

prices_check = pd.read_csv(
    PROCESSED_PATH / "sell_prices_clean.csv"
)

sales_check = pd.read_csv(
    PROCESSED_PATH / "sales_train_validation_clean.csv"
)

print("Calendar:", calendar_check.shape)
print("Prices:  ", prices_check.shape)
print("Sales:   ", sales_check.shape)

print("\nSaved files loaded successfully.")

===== SAVED FILE VALIDATION =====
Calendar: (1969, 14)
Prices:   (6841121, 4)
Sales:    (30490, 1919)

Saved files loaded successfully.
